In [1]:
import pandas as pd
import duckdb

In [2]:
con = duckdb.connect()

In [3]:
v_sales = pd.read_excel("data/hotel_sales_raw_data.xlsx")
con.register('v_sales', v_sales)


In [4]:
# données des ventes brutes
print(v_sales.shape)
v_sales.head(3)

(130566, 18)


,NOM_BOUTIQUE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,NOM_PRODUIT,QUANTITE,PRIX_HT,VAT,PRIX_TTC,TYPE,GAMME,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE
0,Ibis budget Nice,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,TONGS FEMME 100 NOIR,1,5.000000,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9281,25.5
1,Ibis budget Nice,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,CASQUETTE ENFANT -MH100,1,10.000000,20.0,12.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9282,25.5
2,Ibis budget Nice,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,1,24.166667,20.0,29.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9283,25.5


In [6]:
# données des ventes utilisées pour la modélisation
con.sql("""
CREATE OR REPLACE VIEW v_sales_model AS
(
    SELECT 
        * 
    FROM 
        v_sales 
    WHERE 
        (YEAR(DATE) < (SELECT YEAR(MAX(DATE)) FROM v_sales))
        AND
        (TYPE IS NOT NULL)
        AND
        (GAMME IS NOT NULL)
        AND
        (NOM_PRODUIT IS NOT NULL)
        
);
""")

v_sales_model = con.sql("SELECT * FROM v_sales_model").to_df()
print(v_sales_model.shape)
v_sales_model.head(3)

(117440, 18)


,NOM_BOUTIQUE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,NOM_PRODUIT,QUANTITE,PRIX_HT,VAT,PRIX_TTC,TYPE,GAMME,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE
0,Ibis budget Nice,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,TONGS FEMME 100 NOIR,1,5.000000,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9281,25.5
1,Ibis budget Nice,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,CASQUETTE ENFANT -MH100,1,10.000000,20.0,12.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9282,25.5
2,Ibis budget Nice,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,1,24.166667,20.0,29.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9283,25.5


In [7]:
# nombre de mois d'activités de ventes par hotel
con.sql("""
CREATE OR REPLACE VIEW v_nombre_mois AS (
SELECT
    NOM_BOUTIQUE AS hotel_name,
    DATE_DIFF('month', MIN(DATE), MAX(DATE)) + 1 AS nombre_mois
FROM 
    v_sales_model
GROUP BY
    NOM_BOUTIQUE
ORDER BY 
    hotel_name
);
""")

v_nombre_mois = con.sql("SELECT * FROM v_nombre_mois").to_df()
print(v_nombre_mois.shape)
v_nombre_mois.head(3)

(7, 2)


,hotel_name,nombre_mois
0,Ibis budget Nice,29
1,Ibis budget Strasbourg Centre République,15
2,Mercure Paris Boulogne,20


In [8]:
# nombre et montant des ventes globales et par paniers pour chaque hotel
con.sql("""
CREATE OR REPLACE VIEW v_paniers_ventes AS (
SELECT 
    NOM_BOUTIQUE AS hotel_name,

    COUNT(DISTINCT TYPE) AS nombre_types,
    COUNT(DISTINCT GAMME) AS nombre_gammes,
    COUNT(DISTINCT NOM_PRODUIT) AS nombre_produits,

    SUM(QUANTITE) AS nombre_ventes,
    SUM(PRIX_TTC) AS montant_ventes,
    SUM(PRIX_TTC) / SUM(QUANTITE) AS montant_par_vente,

    COUNT(DISTINCT ORDER_ID) AS nombre_paniers,
    SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS nombre_ventes_par_panier,
    SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS montant_ventes_par_panier,
 
FROM
    v_sales_model
GROUP BY
    NOM_BOUTIQUE
ORDER BY 
    hotel_name
);


CREATE OR REPLACE VIEW v_paniers_ventes_mois AS (
SELECT 
    v_paniers_ventes.*,
    nombre_mois,

    nombre_ventes / nombre_mois AS nombre_ventes_par_mois,
    montant_ventes / nombre_mois AS montant_ventes_par_mois,
    nombre_paniers / nombre_mois AS nombre_paniers_par_mois,

FROM 
    v_paniers_ventes
LEFT JOIN
    v_nombre_mois
ON
    v_paniers_ventes.hotel_name = v_nombre_mois.hotel_name
);
""")

v_paniers_ventes_mois = con.sql("SELECT * FROM v_paniers_ventes_mois").to_df()
print(v_paniers_ventes_mois.shape)
v_paniers_ventes_mois.head(3)


(7, 14)


,hotel_name,nombre_types,nombre_gammes,nombre_produits,nombre_ventes,montant_ventes,montant_par_vente,nombre_paniers,nombre_ventes_par_panier,montant_ventes_par_panier,nombre_mois,nombre_ventes_par_mois,montant_ventes_par_mois,nombre_paniers_par_mois
0,Ibis budget Nice,2,11,138,5019.0,17199.580000,3.426894,3620,1.386464,4.751265,29,173.068966,593.088966,124.827586
1,Ibis budget Strasbourg Centre République,1,7,60,7004.0,18598.640000,2.655431,3180,2.202516,5.848629,15,466.933333,1239.909333,212.000000
2,Mercure Paris Boulogne,1,7,78,8676.0,29291.247193,3.376123,6655,1.303681,4.401390,20,433.800000,1464.562360,332.750000


In [9]:
# nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B)
con.sql("""
CREATE OR REPLACE VIEW v_type_paniers_ventes AS (
SELECT 
    NOM_BOUTIQUE AS hotel_name,
    TYPE AS type,

    COUNT(DISTINCT GAMME) AS type_nombre_gammes,
    COUNT(DISTINCT NOM_PRODUIT) AS type_nombre_produits,

    SUM(QUANTITE) AS type_nombre_ventes,
    SUM(PRIX_TTC) AS type_montant_ventes,
    SUM(PRIX_TTC) / SUM(QUANTITE) AS type_montant_par_vente,

    COUNT(DISTINCT ORDER_ID) AS type_nombre_paniers,
    SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS type_nombre_ventes_par_panier,
    SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS type_montant_ventes_par_panier
FROM
    v_sales_model
GROUP BY
    NOM_BOUTIQUE,
    TYPE
ORDER BY 
    hotel_name,
    type
);


CREATE OR REPLACE VIEW v_type_paniers_ventes_mois AS (
SELECT 
    v_type_paniers_ventes.*,
    nombre_mois,

    type_nombre_ventes / nombre_mois AS type_nombre_ventes_par_mois,
    type_montant_ventes / nombre_mois AS type_montant_ventes_par_mois,
    type_nombre_paniers / nombre_mois AS type_nombre_paniers_par_mois,
FROM 
    v_type_paniers_ventes
LEFT JOIN
    v_nombre_mois
ON
    v_type_paniers_ventes.hotel_name = v_nombre_mois.hotel_name
);
""")

v_type_paniers_ventes_mois = con.sql("SELECT * FROM v_type_paniers_ventes_mois").to_df()
print(v_type_paniers_ventes_mois.shape)
v_type_paniers_ventes_mois.head(3)


(12, 14)


,hotel_name,type,type_nombre_gammes,type_nombre_produits,type_nombre_ventes,type_montant_ventes,type_montant_par_vente,type_nombre_paniers,type_nombre_ventes_par_panier,type_montant_ventes_par_panier,nombre_mois,type_nombre_ventes_par_mois,type_montant_ventes_par_mois,type_nombre_paniers_par_mois
0,Ibis budget Nice,F&B,6,41,4327.0,10317.02,2.384336,3082,1.403958,3.347508,29,149.206897,355.759310,106.275862
1,Ibis budget Nice,NON-F&B,5,97,692.0,6882.56,9.945896,555,1.246847,12.401009,29,23.862069,237.329655,19.137931
2,Ibis budget Strasbourg Centre République,F&B,7,60,7004.0,18598.64,2.655431,3180,2.202516,5.848629,15,466.933333,1239.909333,212.000000


In [10]:
# nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme

con.sql("""
CREATE OR REPLACE VIEW v_gamme_paniers_ventes AS (
SELECT 
    NOM_BOUTIQUE AS hotel_name,
    TYPE as type,
    GAMME AS gamme,
    COUNT(DISTINCT NOM_PRODUIT) AS gamme_nombre_produits,

    SUM(QUANTITE) AS gamme_nombre_ventes,
    SUM(PRIX_TTC) AS gamme_montant_ventes,
    SUM(PRIX_TTC) / SUM(QUANTITE) AS gamme_montant_par_vente,

    COUNT(DISTINCT ORDER_ID) AS gamme_nombre_paniers,
    SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS gamme_nombre_ventes_par_panier,
    SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS gamme_montant_ventes_par_panier
FROM
    v_sales_model
GROUP BY
    NOM_BOUTIQUE,
    TYPE,
    GAMME
ORDER BY 
    hotel_name,
    type,
    gamme
);


CREATE OR REPLACE VIEW v_gamme_paniers_ventes_mois AS (
SELECT 
    v_gamme_paniers_ventes.*,
    nombre_mois,

    gamme_nombre_ventes / nombre_mois AS gamme_nombre_ventes_par_mois,
    gamme_montant_ventes / nombre_mois AS gamme_montant_ventes_par_mois,
    gamme_nombre_paniers / nombre_mois AS gamme_nombre_paniers_par_mois,
FROM 
    v_gamme_paniers_ventes
LEFT JOIN
    v_nombre_mois
ON
    v_gamme_paniers_ventes.hotel_name = v_nombre_mois.hotel_name
);
""")

v_gamme_paniers_ventes_mois = con.sql("SELECT * FROM v_gamme_paniers_ventes_mois").to_df()
print(v_gamme_paniers_ventes_mois.shape)
v_gamme_paniers_ventes_mois.head(3)

(73, 14)


,hotel_name,type,gamme,gamme_nombre_produits,gamme_nombre_ventes,gamme_montant_ventes,gamme_montant_par_vente,gamme_nombre_paniers,gamme_nombre_ventes_par_panier,gamme_montant_ventes_par_panier,nombre_mois,gamme_nombre_ventes_par_mois,gamme_montant_ventes_par_mois,gamme_nombre_paniers_par_mois
0,Ibis budget Nice,F&B,FOOD SALEE,4,77.0,170.50,2.214286,74,1.040541,2.304054,29,2.655172,5.879310,2.551724
1,Ibis budget Nice,F&B,FOOD SUCREE,6,44.0,73.80,1.677273,42,1.047619,1.757143,29,1.517241,2.544828,1.448276
2,Ibis budget Nice,F&B,SALTY FOOD (Dry),4,266.0,589.75,2.217105,238,1.117647,2.477941,29,9.172414,20.336207,8.206897


In [11]:
# nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme et produit
con.sql("""
CREATE OR REPLACE VIEW v_produit_paniers_ventes AS (
SELECT 
    NOM_BOUTIQUE AS hotel_name,
    TYPE as type,
    GAMME AS gamme,
    NOM_PRODUIT AS produit,

    SUM(QUANTITE) AS produit_nombre_ventes,
    SUM(PRIX_TTC) AS produit_montant_ventes,
    SUM(PRIX_TTC) / SUM(QUANTITE) AS produit_montant_par_vente,

    COUNT(DISTINCT ORDER_ID) AS produit_nombre_paniers,
    SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS produit_nombre_ventes_par_panier,
    SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS produit_montant_ventes_par_panier
FROM
    v_sales_model
GROUP BY
    NOM_BOUTIQUE,
    TYPE,
    GAMME,
    NOM_PRODUIT
ORDER BY 
    hotel_name,
    type,
    gamme,
    produit
);


CREATE OR REPLACE VIEW v_produit_paniers_ventes_mois AS (
SELECT 
    v_produit_paniers_ventes.*,
    nombre_mois,

    produit_nombre_ventes / nombre_mois AS produit_nombre_ventes_par_mois,
    produit_montant_ventes / nombre_mois AS produit_montant_ventes_par_mois,
    produit_nombre_paniers / nombre_mois AS produit_nombre_paniers_par_mois,
FROM 
    v_produit_paniers_ventes
LEFT JOIN
    v_nombre_mois
ON
    v_produit_paniers_ventes.hotel_name = v_nombre_mois.hotel_name
);
""")


v_produit_paniers_ventes_mois = con.sql("SELECT * FROM v_produit_paniers_ventes_mois").to_df()
print(v_produit_paniers_ventes_mois.shape)
v_produit_paniers_ventes_mois.head(3)

(1246, 14)


,hotel_name,type,gamme,produit,produit_nombre_ventes,produit_montant_ventes,produit_montant_par_vente,produit_nombre_paniers,produit_nombre_ventes_par_panier,produit_montant_ventes_par_panier,nombre_mois,produit_nombre_ventes_par_mois,produit_montant_ventes_par_mois,produit_nombre_paniers_par_mois
0,Ibis budget Nice,F&B,FOOD SALEE,CHIPS DE POULET ROTI 45G,65.0,144.5,2.223077,65,1.0,2.223077,29,2.241379,4.982759,2.241379
1,Ibis budget Nice,F&B,FOOD SALEE,CHIPS LAY'S SEL 45G,2.0,4.0,2.000000,2,1.0,2.000000,29,0.068966,0.137931,0.068966
2,Ibis budget Nice,F&B,FOOD SALEE,DORITOS NACHO CHEESE 44G,6.0,12.0,2.000000,6,1.0,2.000000,29,0.206897,0.413793,0.206897


In [12]:
# vue template pour la modélisation
con.sql("""
CREATE OR REPLACE VIEW v_sales_model_template AS (
SELECT
    *
FROM
    v_produit_paniers_ventes

INNER JOIN
    v_gamme_paniers_ventes
ON
    (v_produit_paniers_ventes.hotel_name = v_gamme_paniers_ventes.hotel_name)
    AND
    (v_produit_paniers_ventes.type = v_gamme_paniers_ventes.type)
    AND
    (v_produit_paniers_ventes.gamme = v_gamme_paniers_ventes.gamme)

INNER JOIN
    v_type_paniers_ventes
ON
    (v_type_paniers_ventes.hotel_name = v_gamme_paniers_ventes.hotel_name)
    AND
    (v_type_paniers_ventes.type = v_gamme_paniers_ventes.type)

INNER JOIN
    v_paniers_ventes
ON
    (v_paniers_ventes.hotel_name = v_gamme_paniers_ventes.hotel_name)
)
;
""")

v_sales_model_template = con.sql("SELECT * FROM v_sales_model_template").to_df()
print(v_sales_model_template.shape)
v_sales_model_template.head(3)

(1246, 40)


,hotel_name,type,gamme,produit,produit_nombre_ventes,produit_montant_ventes,produit_montant_par_vente,produit_nombre_paniers,produit_nombre_ventes_par_panier,produit_montant_ventes_par_panier,...,hotel_name_3,nombre_types,nombre_gammes,nombre_produits,nombre_ventes,montant_ventes,montant_par_vente,nombre_paniers,nombre_ventes_par_panier,montant_ventes_par_panier
0,Ibis budget Nice,F&B,FOOD SALEE,CHIPS DE POULET ROTI 45G,65.0,144.5,2.223077,65,1.0,2.223077,...,Ibis budget Nice,2,11,138,5019.0,17199.58,3.426894,3620,1.386464,4.751265
1,Ibis budget Nice,F&B,FOOD SALEE,CHIPS LAY'S SEL 45G,2.0,4.0,2.000000,2,1.0,2.000000,...,Ibis budget Nice,2,11,138,5019.0,17199.58,3.426894,3620,1.386464,4.751265
2,Ibis budget Nice,F&B,FOOD SALEE,DORITOS NACHO CHEESE 44G,6.0,12.0,2.000000,6,1.0,2.000000,...,Ibis budget Nice,2,11,138,5019.0,17199.58,3.426894,3620,1.386464,4.751265


In [30]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)                # Force le scrollbar horizontal
pd.set_option('display.expand_frame_repr', True)

In [33]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)                # Force le scrollbar horizontal
pd.set_option('display.expand_frame_repr', True)

class SalesPrep():
    def __init__(self, sales_path : str):
        self.sales_path = sales_path

        self._con = None

        self._v_sales_model_template = None


    @property
    def con(self):
        if(self._con is None):
            self._con = duckdb.connect()

        return self._con
    
        
    @property
    def v_sales_model_template(self):
        if(self._v_sales_model_template is None):
            self._v_sales_model_template = self.view_df("v_sales_model_template")
        return self._v_sales_model_template


    def view_exists(self, view_name : str):
        exists = self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.views 
            WHERE table_name = '{view_name}'
        """).fetchone()[0]

        return exists
    

    def create_if_not_exists_view(self, view_name : str):
        if(not self.view_exists(view_name)):
            getattr(self, f"create_or_replace_view_{view_name}")()
    
    
    def create_if_not_exists_views(self, view_names : list[str]):
        for view_name in view_names:
            self.create_if_not_exists_view(view_name)


    def view_df(self, view_name : str):
            self.create_if_not_exists_view(view_name)
            df = self.con.sql(f"SELECT * FROM {view_name}").df()
            return df


    def create_or_replace_view_v_sales(self):
        # données des ventes brutes
        v_sales = pd.read_excel(self.sales_path)
        self.con.register('v_sales', v_sales)


    def create_or_replace_view_v_sales_model(self):
        self.create_if_not_exists_view("v_sales")

        # données des ventes utilisées pour la modélisation
        self.con.sql("""
            CREATE OR REPLACE VIEW v_sales_model AS
            (
                SELECT 
                    * 
                FROM 
                    v_sales 
                WHERE 
                    (YEAR(DATE) < (SELECT YEAR(MAX(DATE)) FROM v_sales))
                    AND
                    (TYPE IS NOT NULL)
                    AND
                    (GAMME IS NOT NULL)
                    AND
                    (NOM_PRODUIT IS NOT NULL)
                    
            );
        """)

        # v_sales_model = con.sql("SELECT * FROM v_sales_model").to_df()
        # print(v_sales_model.shape)
        # display(v_sales_model.head(3))

    def create_or_replace_view_v_nombre_mois(self):
        self.create_if_not_exists_view("v_sales_model")

        # nombre de mois d'activités de ventes par hotel
        self.con.sql("""
            CREATE OR REPLACE VIEW v_nombre_mois AS (
            SELECT
                NOM_BOUTIQUE AS hotel_name,
                DATE_DIFF('month', MIN(DATE), MAX(DATE)) + 1 AS nombre_mois
            FROM 
                v_sales_model
            GROUP BY
                NOM_BOUTIQUE
            ORDER BY 
                hotel_name
        );
        """)

        # v_nombre_mois = con.sql("SELECT * FROM v_nombre_mois").to_df()
        # print(v_nombre_mois.shape)
        # display(v_nombre_mois.head(3))


    def create_or_replace_view_v_paniers_ventes(self):
        self.create_if_not_exists_views(["v_sales_model", "v_nombre_mois"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel
        self.con.sql("""
            CREATE OR REPLACE VIEW v_paniers_ventes AS (
            SELECT 
                NOM_BOUTIQUE AS hotel_name,

                COUNT(DISTINCT TYPE) AS nombre_types,
                COUNT(DISTINCT GAMME) AS nombre_gammes,
                COUNT(DISTINCT NOM_PRODUIT) AS nombre_produits,

                SUM(QUANTITE) AS nombre_ventes,
                SUM(PRIX_TTC) AS montant_ventes,
                SUM(PRIX_TTC) / SUM(QUANTITE) AS montant_par_vente,

                COUNT(DISTINCT ORDER_ID) AS nombre_paniers,
                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS montant_ventes_par_panier,
            
            FROM
                v_sales_model
            GROUP BY
                NOM_BOUTIQUE
            ORDER BY 
                hotel_name
            );


            CREATE OR REPLACE VIEW v_paniers_ventes_mois AS (
            SELECT 
                v_paniers_ventes.*,
                nombre_mois,

                nombre_ventes / nombre_mois AS nombre_ventes_par_mois,
                montant_ventes / nombre_mois AS montant_ventes_par_mois,
                nombre_paniers / nombre_mois AS nombre_paniers_par_mois,

            FROM 
                v_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_paniers_ventes.hotel_name = v_nombre_mois.hotel_name
            );
        """)

        # v_paniers_ventes_mois = con.sql("SELECT * FROM v_paniers_ventes_mois").to_df()
        # print(v_paniers_ventes_mois.shape)
        # display(v_paniers_ventes_mois.head(3))


    def create_or_replace_view_v_type_paniers_ventes(self):
        self.create_if_not_exists_views(["v_sales_model", "v_nombre_mois"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B)
        self.con.sql("""
            CREATE OR REPLACE VIEW v_type_paniers_ventes AS (
            SELECT 
                NOM_BOUTIQUE AS hotel_name,
                TYPE AS type,

                COUNT(DISTINCT GAMME) AS type_nombre_gammes,
                COUNT(DISTINCT NOM_PRODUIT) AS type_nombre_produits,

                SUM(QUANTITE) AS type_nombre_ventes,
                SUM(PRIX_TTC) AS type_montant_ventes,
                SUM(PRIX_TTC) / SUM(QUANTITE) AS type_montant_par_vente,

                COUNT(DISTINCT ORDER_ID) AS type_nombre_paniers,
                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS type_nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS type_montant_ventes_par_panier
            FROM
                v_sales_model
            GROUP BY
                NOM_BOUTIQUE,
                TYPE
            ORDER BY 
                hotel_name,
                type
            );


            CREATE OR REPLACE VIEW v_type_paniers_ventes_mois AS (
            SELECT 
                v_type_paniers_ventes.*,
                nombre_mois,

                type_nombre_ventes / nombre_mois AS type_nombre_ventes_par_mois,
                type_montant_ventes / nombre_mois AS type_montant_ventes_par_mois,
                type_nombre_paniers / nombre_mois AS type_nombre_paniers_par_mois,
            FROM 
                v_type_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_type_paniers_ventes.hotel_name = v_nombre_mois.hotel_name
            );
        """)

        # v_type_paniers_ventes_mois = con.sql("SELECT * FROM v_type_paniers_ventes_mois").to_df()
        # print(v_type_paniers_ventes_mois.shape)
        # display(v_type_paniers_ventes_mois.head(3))
    
    def create_or_replace_view_v_gamme_paniers_ventes(self):
        self.create_if_not_exists_views(["v_sales_model", "v_nombre_mois"])
                    
        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme
        self.con.sql("""
            CREATE OR REPLACE VIEW v_gamme_paniers_ventes AS (
            SELECT 
                NOM_BOUTIQUE AS hotel_name,
                TYPE as type,
                GAMME AS gamme,
                COUNT(DISTINCT NOM_PRODUIT) AS gamme_nombre_produits,

                SUM(QUANTITE) AS gamme_nombre_ventes,
                SUM(PRIX_TTC) AS gamme_montant_ventes,
                SUM(PRIX_TTC) / SUM(QUANTITE) AS gamme_montant_par_vente,

                COUNT(DISTINCT ORDER_ID) AS gamme_nombre_paniers,
                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS gamme_nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS gamme_montant_ventes_par_panier
            FROM
                v_sales_model
            GROUP BY
                NOM_BOUTIQUE,
                TYPE,
                GAMME
            ORDER BY 
                hotel_name,
                type,
                gamme
            );


            CREATE OR REPLACE VIEW v_gamme_paniers_ventes_mois AS (
            SELECT 
                v_gamme_paniers_ventes.*,
                nombre_mois,

                gamme_nombre_ventes / nombre_mois AS gamme_nombre_ventes_par_mois,
                gamme_montant_ventes / nombre_mois AS gamme_montant_ventes_par_mois,
                gamme_nombre_paniers / nombre_mois AS gamme_nombre_paniers_par_mois,
            FROM 
                v_gamme_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_gamme_paniers_ventes.hotel_name = v_nombre_mois.hotel_name
            );
        """)

        # v_gamme_paniers_ventes_mois = con.sql("SELECT * FROM v_gamme_paniers_ventes_mois").to_df()
        # print(v_gamme_paniers_ventes_mois.shape)
        # display(v_gamme_paniers_ventes_mois.head(3))

    def create_or_replace_view_v_produit_paniers_ventes(self):
        self.create_if_not_exists_views(["v_sales_model", "v_nombre_mois"])
                    
        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme et produit
        self.con.sql("""
            CREATE OR REPLACE VIEW v_produit_paniers_ventes AS (
            SELECT 
                NOM_BOUTIQUE AS hotel_name,
                TYPE as type,
                GAMME AS gamme,
                NOM_PRODUIT AS produit,

                SUM(QUANTITE) AS produit_nombre_ventes,
                SUM(PRIX_TTC) AS produit_montant_ventes,
                SUM(PRIX_TTC) / SUM(QUANTITE) AS produit_montant_par_vente,

                COUNT(DISTINCT ORDER_ID) AS produit_nombre_paniers,
                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS produit_nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS produit_montant_ventes_par_panier
            FROM
                v_sales_model
            GROUP BY
                NOM_BOUTIQUE,
                TYPE,
                GAMME,
                NOM_PRODUIT
            ORDER BY 
                hotel_name,
                type,
                gamme,
                produit
            );


            CREATE OR REPLACE VIEW v_produit_paniers_ventes_mois AS (
            SELECT 
                v_produit_paniers_ventes.*,
                nombre_mois,

                produit_nombre_ventes / nombre_mois AS produit_nombre_ventes_par_mois,
                produit_montant_ventes / nombre_mois AS produit_montant_ventes_par_mois,
                produit_nombre_paniers / nombre_mois AS produit_nombre_paniers_par_mois,
            FROM 
                v_produit_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_produit_paniers_ventes.hotel_name = v_nombre_mois.hotel_name
            );
        """)


        # v_produit_paniers_ventes_mois = con.sql("SELECT * FROM v_produit_paniers_ventes_mois").to_df()
        # print(v_produit_paniers_ventes_mois.shape)
        # display(v_produit_paniers_ventes_mois.head(3))

    def create_or_replace_view_v_sales_model_template(self):
        self.create_if_not_exists_views(["v_produit_paniers_ventes", "v_gamme_paniers_ventes", "v_type_paniers_ventes", "v_paniers_ventes"])
                    
        # vue template pour la modélisation
        self.con.sql("""
            CREATE OR REPLACE VIEW v_sales_model_template AS (
                SELECT 
                    *
                FROM 
                    v_produit_paniers_ventes
                
                INNER JOIN 
                    v_gamme_paniers_ventes
                USING 
                    (hotel_name, type, gamme)
                
                INNER JOIN 
                    v_type_paniers_ventes
                USING 
                    (hotel_name, type)
                
                INNER JOIN 
                    v_paniers_ventes
                USING 
                    (hotel_name)
            );
        """)

        # v_sales_model_template = con.sql("SELECT * FROM v_sales_model_template").to_df()
        # print(v_sales_model_template.shape)
        # display(v_sales_model_template.head(3))



In [34]:
self = SalesPrep("data/hotel_sales_raw_data.xlsx")

In [35]:
self.v_sales_model_template.head(3)

,hotel_name,type,gamme,produit,produit_nombre_ventes,produit_montant_ventes,produit_montant_par_vente,produit_nombre_paniers,produit_nombre_ventes_par_panier,produit_montant_ventes_par_panier,gamme_nombre_produits,gamme_nombre_ventes,gamme_montant_ventes,gamme_montant_par_vente,gamme_nombre_paniers,gamme_nombre_ventes_par_panier,gamme_montant_ventes_par_panier,type_nombre_gammes,type_nombre_produits,type_nombre_ventes,type_montant_ventes,type_montant_par_vente,type_nombre_paniers,type_nombre_ventes_par_panier,type_montant_ventes_par_panier,nombre_types,nombre_gammes,nombre_produits,nombre_ventes,montant_ventes,montant_par_vente,nombre_paniers,nombre_ventes_par_panier,montant_ventes_par_panier
0,Ibis budget Nice,F&B,FOOD SALEE,CHIPS DE POULET ROTI 45G,65.0,144.5,2.223077,65,1.0,2.223077,4,77.0,170.5,2.214286,74,1.040541,2.304054,6,41,4327.0,10317.02,2.384336,3082,1.403958,3.347508,2,11,138,5019.0,17199.58,3.426894,3620,1.386464,4.751265
1,Ibis budget Nice,F&B,FOOD SALEE,CHIPS LAY'S SEL 45G,2.0,4.0,2.000000,2,1.0,2.000000,4,77.0,170.5,2.214286,74,1.040541,2.304054,6,41,4327.0,10317.02,2.384336,3082,1.403958,3.347508,2,11,138,5019.0,17199.58,3.426894,3620,1.386464,4.751265
2,Ibis budget Nice,F&B,FOOD SALEE,DORITOS NACHO CHEESE 44G,6.0,12.0,2.000000,6,1.0,2.000000,4,77.0,170.5,2.214286,74,1.040541,2.304054,6,41,4327.0,10317.02,2.384336,3082,1.403958,3.347508,2,11,138,5019.0,17199.58,3.426894,3620,1.386464,4.751265


In [37]:
produit = "CHIPS DE POULET ROTI 45G"

self.con.sql(f"""
SELECT 
    * 
FROM 
    v_sales_model_template 
WHERE
    produit = '{produit}'
""")

┌──────────────────┬─────────┬──────────────────┬──────────────────────────┬───────────────────────┬────────────────────────┬───────────────────────────┬────────────────────────┬──────────────────────────────────┬───────────────────────────────────┬───────────────────────┬─────────────────────┬──────────────────────┬─────────────────────────┬──────────────────────┬────────────────────────────────┬─────────────────────────────────┬────────────────────┬──────────────────────┬────────────────────┬─────────────────────┬────────────────────────┬─────────────────────┬───────────────────────────────┬────────────────────────────────┬──────────────┬───────────────┬─────────────────┬───────────────┬────────────────────┬───────────────────┬────────────────┬──────────────────────────┬───────────────────────────┐
│    hotel_name    │  type   │      gamme       │         produit          │ produit_nombre_ventes │ produit_montant_ventes │ produit_montant_par_vente │ produit_nombre_paniers │ produit_n

In [ ]:
nombre_paniers
montant_ventes
nombre_ventes
nombre_produits
nombre_gammes
nombre_types

type_montant_ventes
type_nombre_ventes
type_nombre_produits
type_nombre_gammes

gamme_nombre_paniers
gamme_montant_ventes
gamme_nombre_ventes
gamme_nombre_produits
